In [ ]:
from rdkit import Chem
from rdkit.Chem import AllChem
import subprocess

Step 1: Generate Molecule Geometry (RDKit)

In [ ]:
smiles = "c1ccncc1c2ccccc2"  # azaacene
mol = Chem.MolFromSmiles(smiles)
mol = Chem.AddHs(mol)
AllChem.EmbedMolecule(mol, randomSeed=42)
AllChem.UFFOptimizeMolecule(mol)

xyz_file = "azaacene.xyz"
with open(xyz_file, "w") as f:
    conf = mol.GetConformer()
    f.write(f"{mol.GetNumAtoms()}\n\n")
    for atom in mol.GetAtoms():
        pos = conf.GetAtomPosition(atom.GetIdx())
        f.write(f"{atom.GetSymbol()} {pos.x:.6f} {pos.y:.6f} {pos.z:.6f}\n")

Step 2: Prepare ORCA Input Files

In [ ]:
def write_orca_input(xyz_file, inp_file, charge, multiplicity,
                     method="PBE0", basis="def2-TZVP", optimize=True):
    """
    Generate an ORCA input file from an xyz geometry.
    - xyz_file: path to .xyz file with coordinates
    - inp_file: output .inp filename
    - charge: molecular charge (int)
    - multiplicity: spin multiplicity (int)
    - method: DFT functional (default PBE0)
    - basis: basis set (default def2-TZVP)
    - optimize: if True, add 'Opt' keyword, else single-point
    """
    # Read coordinates from xyz file
    with open(xyz_file) as f:
        lines = f.readlines()[2:]
    coords = "".join(lines)

    # Choose job type
    job_type = "Opt TightSCF" if optimize else "TightSCF"

    # Build ORCA input content
    content = f"""! {method} {basis} {job_type}
* xyz {charge} {multiplicity}
{coords}*
"""
    with open(inp_file, "w") as f:
        f.write(content)


# Neutral optimization
write_orca_input("azaacene.xyz", "neutral.inp", charge=0, multiplicity=1, optimize=True)

# Cation optimization 
write_orca_input("azaacene.xyz", "cation.inp", charge=+1, multiplicity=2, optimize=True)


Step 3: Run ORCA from Python

In [ ]:
subprocess.run("orca neutral.inp > neutral.out", shell=True, check=True)
subprocess.run("orca cation.inp > cation.out", shell=True, check=True)

In [ ]:
def extract_final_xyz(out_file, xyz_file):
    """
    Extract the final optimized geometry from an ORCA .out file
    and save it as an .xyz file.
    """
    with open(out_file) as f:
        lines = f.readlines()

    # Find the last occurrence of 'CARTESIAN COORDINATES (ANGSTROEM)'
    start = None
    for i, line in enumerate(lines):
        if "CARTESIAN COORDINATES (ANGSTROEM)" in line:
            start = i + 2  # skip header lines

    if start is None:
        raise ValueError("No optimized coordinates found in output file.")

    coords = []
    for line in lines[start:]:
        if line.strip() == "":
            break
        coords.append(line.strip())

    # Write to xyz file
    with open(xyz_file, "w") as f:
        f.write(f"{len(coords)}\n")
        f.write(f"Extracted from {out_file}\n")
        for line in coords:
            f.write(line + "\n")

extract_final_xyz("neutral.out", "neutral_opt.xyz")
extract_final_xyz("cation.out", "cation_opt.xyz")

In [ ]:
# Neutral geometry as cation (single-point)
write_orca_input("neutral_opt.xyz", "neutral_sp_cation.inp", charge=+1, multiplicity=2, optimize=False)
write_orca_input("cation_opt.xyz", "cation_sp_neutral.inp", charge=0, multiplicity=1, optimize=False)

subprocess.run("orca neutral_sp_cation.inp > neutral_sp_cation.out", shell=True, check=True)
subprocess.run("orca cation_sp_neutral.inp > cation_sp_neutral.out", shell=True, check=True)

Step 4: Extract energy values and calculate HRE

In [ ]:
def extract_final_energy(filename):
    """
    Read the ORCA output file line by line and extract the FINAL SINGLE POINT ENERGY value.
    """
    with open(filename, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            if "FINAL SINGLE POINT ENERGY" in line:
                # print("Matched line:", repr(line))
                return float(line.strip().split()[-1])
    return None

ENN = extract_final_energy("neutral.out")
ECC = extract_final_energy("cation.out")
ENC = extract_final_energy("neutral_sp_cation.out")
ECN = extract_final_energy("cation_sp_neutral.out")

hre = (ENC - ECC) + (ECN - ENN)
print("HRE (Hartrees):", hre)
print("HRE (eV):", hre * 27.2114)

-478.930393271211 -478.622902682224 -478.622902682224 -478.930393271211
